In [1]:
import sys

sys.path.append("..")
import numpy as np


from model.models import FMCond
from model.networks import MLP
import trimesh
from util.matching_utils import *
from util.mesh_utils import *
import os
import torch

import sys

from util.matching_utils import (
    compute_p2p_with_geomdist,
)
from util.mesh_utils import normalize_mesh, pc_normalize
from geomfum.shape.mesh import TriangleMesh
from geomfum.metric.mesh import HeatDistanceMetric
import potpourri3d as pp3d

device = "cuda:0"
torch.cuda.set_device(device)

PLOT = True

In [2]:
SOURCE = "tr_reg_080"
TARGET = "tr_reg_081"
INPUT_DIR = "../../data/faust/test_set/"
CORR_DIR = "/nonesiste/"

In [3]:
mesh1_path = os.path.join(INPUT_DIR, f"{SOURCE}.off")
mesh2_path = os.path.join(INPUT_DIR, f"{TARGET}.off")


mesh1 = trimesh.load(mesh1_path, process=False)
mesh2 = trimesh.load(mesh2_path, process=False)
if len(mesh1.faces) > 0:
    mesh1 = normalize_mesh(trimesh.load(mesh1_path, process=False))
    mesh2 = normalize_mesh(trimesh.load(mesh2_path, process=False))
else:
    mesh1.vertices = pc_normalize(mesh1.vertices)
    mesh2.vertices = pc_normalize(mesh2.vertices)

# Load correspondences

corr_a_path = os.path.join(CORR_DIR, f"{SOURCE}.vts")
corr_b_path = os.path.join(CORR_DIR, f"{TARGET}.vts")

if not os.path.exists(corr_a_path) or not os.path.exists(corr_b_path):
    print("Warning: Correspondence files not found. Using Identity.")
    corr_a = np.arange(len(mesh1.vertices))
    corr_b = np.arange(len(mesh2.vertices))
else:
    corr_a = np.array(np.loadtxt(corr_a_path, dtype=np.int32))
    corr_b = np.array(np.loadtxt(corr_b_path, dtype=np.int32))

    # Apply offset if needed (for datasets like SMAL that are 1-indexed)
    if CORR_DIR == "../datasets/meshes/SMAL_r/corres/":
        corr_a = corr_a - 1
        corr_b = corr_b - 1

# Target vertices for evaluation
v2 = torch.tensor(mesh2.vertices).float().to(device)
if len(mesh2.faces) > 0:
    # Try to load the distance matrix from cache if it exists
    mesh_gf = TriangleMesh(np.array(mesh2.vertices), np.array(mesh2.faces))
    heat = HeatDistanceMetric(mesh_gf)
    dist = heat.dist_matrix()
else:
    solver = pp3d.PointCloudHeatSolver(np.array(mesh2.vertices))
    distances = []
    for idx in range(len(mesh2.vertices)):
        distances.append(solver.compute_distance(idx))
    dist = np.array(distances)

# Load features
source_feature_path = os.path.join(
    "../../experiments/faust/faust_5_landmarks_exp/", f"{SOURCE}/features.txt"
)
target_feature_path = os.path.join(
    "../../experiments/faust/faust_5_landmarks_exp/", f"{TARGET}/features.txt"
)

source_features = torch.tensor(np.loadtxt(source_feature_path).astype(np.float32)).to(
    device
)
target_features = torch.tensor(np.loadtxt(target_feature_path).astype(np.float32)).to(
    device
)

# Load models

source_checkpoint_path = os.path.join(
    "../../experiments/faust/faust_5_landmarks_exp/", f"{SOURCE}/checkpoint-9999.pth"
)
target_checkpoint_path = os.path.join(
    "../../experiments/faust/faust_5_landmarks_exp/", f"{TARGET}/checkpoint-9999.pth"
)

source_model = FMCond(
    channels=5,
    network=MLP(channels=5).to(device),
)
source_model.to(device)
source_model.load_state_dict(
    torch.load(source_checkpoint_path, map_location=device, weights_only=False)[
        "model"
    ],
    strict=True,
)

target_model = FMCond(
    channels=5,
    network=MLP(channels=5).to(device),
)
target_model.to(device)
target_model.load_state_dict(
    torch.load(target_checkpoint_path, map_location=device, weights_only=False)[
        "model"
    ],
    strict=True,
)

source_model.eval()
target_model.eval()

p2p = compute_p2p_with_geomdist(
    source_features, target_features, source_model, target_model
)


/home/ubuntu/giulio_vigano/flowmatching_proj/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
source_feature_path

'../../experiments/faust/faust_5_landmarks_exp/tr_reg_080/features.txt'

In [5]:
from util.plot import start_end_subplot

In [ ]:
start_end_subplot(
    torch.tensor(mesh1.vertices), torch.tensor(mesh2.vertices[p2p]), show=True
)